# 🌱➡️💻 임베딩(Embedding): 처음 개념부터 코드 실습까지

> 이 노트북의 목표는 **완전 초보 개념 이해 → 작은 숫자 실험 → 문장 검색 코드 구현**까지 자연스럽게 올라가며, 임베딩이 왜 “의미를 숫자로 다루는 방법”인지 체감하는 것입니다.

📊 함께 볼 슬라이드: [embedding.pptx](../slides/embedding.pptx)

---

## 오늘의 큰 그림

AI는 글의 의미를 직접 “느끼는” 것이 아니라, 글을 숫자 벡터로 바꿔 계산합니다.

```text
"시험"   → [0.95, 0.05, 0.00, 0.00]
"테스트" → [0.92, 0.04, 0.00, 0.00]
"축구"   → [0.02, 0.95, 0.00, 0.00]
```

숫자의 방향이 비슷하면 의미도 비슷하다고 볼 수 있습니다.

**중요한 주의**: 이 노트북의 임베딩 숫자는 원리를 이해하기 위해 사람이 만든 **toy embedding**, 즉 설명용 가짜 임베딩입니다. 실제 임베딩 모델은 많은 데이터에서 이 숫자를 학습합니다.

**핵심 메시지**: *임베딩은 AI가 의미를 계산할 수 있도록 단어, 문장, 문서를 숫자 좌표로 바꾸는 방법입니다.*

이 자료는 각 개념을 설명한 뒤 **“ChatGPT에게 확인질문”** 블록을 제공합니다. 설명을 읽은 다음 해당 질문을 ChatGPT에 넣어보면, 같은 개념을 다른 비유와 예시로 다시 확인할 수 있습니다.


## 1단계: 문제 상황 — 컴퓨터는 글의 뜻을 모른다

사람은 "시험"과 "테스트"가 비슷한 말이라는 것을 쉽게 압니다. 하지만 컴퓨터는 문자를 그대로 보면 의미보다 문자열만 봅니다.

그래서 AI는 보통 다음 흐름으로 텍스트를 처리합니다.

1. 텍스트를 작은 조각으로 나눈다: 토큰화
2. 조각이나 문장을 숫자 벡터로 바꾼다: 임베딩
3. 벡터끼리 가까운지 계산한다: 유사도
4. 가까운 문서나 답변 후보를 찾는다: 검색, 추천, RAG

먼저 작은 실험용 데이터셋을 준비합니다.

### 💬 ChatGPT에게 확인질문
```text
고등학생에게 설명하듯이 임베딩이 왜 필요한지 알려주세요.
컴퓨터가 글의 뜻을 바로 이해하지 못한다는 점과, 글을 숫자 벡터로 바꾸는 이유를 쉬운 비유로 설명해주세요.
```


In [1]:
# 통합 노트북에서 사용할 작은 데이터셋
words = [
    "시험", "테스트", "공부", "계획", "불안", "발표", "질문",
    "학교", "교실", "선생님", "친구", "동아리",
    "축구", "농구", "운동", "운동장", "골대", "연습",
    "라면", "떡볶이", "급식", "도시락",
    "버스", "지하철", "자동차", "등교", "지각",
]

sentences = [
    "오늘 시험이 너무 어려웠다",
    "이번 테스트는 정말 힘들었다",
    "시험 불안을 줄이려고 공부 계획을 세웠다",
    "교실에서 선생님께 질문했다",
    "발표 준비를 친구와 함께 연습했다",
    "오늘 운동장에서 축구를 했다",
    "농구 골대 앞에서 친구들과 연습했다",
    "점심으로 라면과 떡볶이를 먹었다",
    "버스를 타고 학교에 갔다",
]

documents = {
    "공부 스트레스": "시험 기간에는 계획을 작게 나누고 쉬는 시간을 정하면 불안을 줄일 수 있다.",
    "발표 준비": "발표를 잘하려면 핵심 문장을 먼저 정하고 친구 앞에서 소리 내어 연습한다.",
    "운동 습관": "축구나 농구처럼 좋아하는 운동을 정해 꾸준히 하면 체력 관리에 도움이 된다.",
    "등교 방법": "버스나 지하철 시간을 미리 확인하면 지각을 줄일 수 있다.",
    "점심 추천": "급식이 맞지 않는 날에는 간단하고 균형 잡힌 도시락을 준비할 수 있다.",
}

print(f"📚 단어 수: {len(words)}개")
print(f"📝 문장 수: {len(sentences)}개")
print(f"📄 문서 수: {len(documents)}개")
print("\n예시 문장:")
for sentence in sentences[:4]:
    print("-", sentence)


📚 단어 수: 27개
📝 문장 수: 9개
📄 문서 수: 5개

예시 문장:
- 오늘 시험이 너무 어려웠다
- 이번 테스트는 정말 힘들었다
- 시험 불안을 줄이려고 공부 계획을 세웠다
- 교실에서 선생님께 질문했다


## 2단계: 토크나이저와 임베딩 — 나누기와 의미 표현은 다르다

토크나이저와 임베딩은 자주 함께 나오지만 역할이 다릅니다.

| 구분 | 하는 일 | 예시 |
|---|---|---|
| 토크나이저 | 문장을 작은 조각으로 나눔 | "나는 학교에 간다" → ["나는", "학교에", "간다"] |
| 임베딩 | 조각이나 문장을 숫자 좌표로 바꿈 | "학교" → [0.8, 0.1, 0.0, 0.0] |

즉, **토크나이저는 자르는 역할**, **임베딩은 의미를 숫자로 표현하는 역할**입니다.

### 💬 ChatGPT에게 확인질문
```text
토크나이저와 임베딩의 차이를 표로 정리해주세요.
각각의 역할, 입력, 출력, 쉬운 비유를 포함해서 고등학생 눈높이로 설명해주세요.
```


In [2]:
import re
from collections import Counter


def simple_tokenize(text):
    cleaned = re.sub(r"[^0-9A-Za-z가-힣\s]", "", text)
    return cleaned.split()

sample = "오늘 시험이 너무 어려웠다! 그래도 다시 공부하자."
print("✂️ 토큰화 예시")
print("입력:", sample)
print("토큰:", simple_tokenize(sample))


✂️ 토큰화 예시
입력: 오늘 시험이 너무 어려웠다! 그래도 다시 공부하자.
토큰: ['오늘', '시험이', '너무', '어려웠다', '그래도', '다시', '공부하자']


## 3단계: 가장 단순한 벡터화 — 단어 개수 세기

임베딩으로 가기 전, 먼저 텍스트를 숫자로 바꾸는 가장 단순한 방법을 봅니다.

어휘가 `[시험, 공부, 축구]`라면,

```text
"시험 공부" → [1, 1, 0]
"축구 축구" → [0, 0, 2]
```

이 방식은 진짜 의미 임베딩은 아니지만, 텍스트를 숫자 목록으로 바꾼다는 핵심 감각을 잡는 데 좋습니다.

### 💬 ChatGPT에게 확인질문
```text
단어 개수 벡터가 무엇인지 쉬운 예시로 설명해주세요.
그리고 단어 개수 벡터가 진짜 의미 임베딩과 어떻게 다른지도 알려주세요.
```


In [3]:
def build_vocabulary(texts):
    return sorted({token for text in texts for token in simple_tokenize(text)})


def count_vector(text, vocabulary):
    counts = Counter(simple_tokenize(text))
    return [counts[word] for word in vocabulary]

vocabulary = build_vocabulary(sentences + list(documents.values()))
example = "오늘 시험이 너무 어려웠다"
vector = count_vector(example, vocabulary)
nonzero = [(word, value) for word, value in zip(vocabulary, vector) if value]

print(f"🧱 어휘 크기: {len(vocabulary)}개")
print("어휘 일부:", vocabulary[:12])
print("\n문장:", example)
print("0이 아닌 벡터 값:", nonzero)


🧱 어휘 크기: 81개
어휘 일부: ['간단하고', '갔다', '계획을', '골대', '공부', '관리에', '교실에서', '균형', '급식이', '기간에는', '꾸준히', '나누고']

문장: 오늘 시험이 너무 어려웠다
0이 아닌 벡터 값: [('너무', 1), ('시험이', 1), ('어려웠다', 1), ('오늘', 1)]


## 4단계: 유사도 — 가까운 벡터를 점수로 계산하기

벡터가 생기면 두 텍스트가 얼마나 비슷한지 계산할 수 있습니다.

여기서는 **코사인 유사도**를 사용합니다.
- 1에 가까울수록 매우 비슷함
- 0에 가까울수록 관련이 적음

단어 개수 방식은 같은 단어가 겹칠 때 강합니다. 하지만 "시험"과 "테스트"처럼 단어가 달라도 뜻이 비슷한 경우는 잘 못 잡을 수 있습니다.

### 💬 ChatGPT에게 확인질문
```text
코사인 유사도가 무엇인지 수학을 어려워하는 고등학생도 이해할 수 있게 설명해주세요.
두 문장의 벡터 방향이 비슷하다는 말이 무슨 뜻인지 비유와 함께 알려주세요.
```


In [4]:
from math import sqrt


def cosine_similarity(vec_a, vec_b):
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = sqrt(sum(a * a for a in vec_a))
    norm_b = sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

query = "시험 공부가 너무 힘들다"
query_vector = count_vector(query, vocabulary)

print("📏 단어 개수 벡터 기반 유사도")
print("기준 문장:", query)
for sentence in sentences:
    score = cosine_similarity(query_vector, count_vector(sentence, vocabulary))
    print(f"{score:.3f} | {sentence}")


📏 단어 개수 벡터 기반 유사도
기준 문장: 시험 공부가 너무 힘들다
0.354 | 오늘 시험이 너무 어려웠다
0.000 | 이번 테스트는 정말 힘들었다
0.289 | 시험 불안을 줄이려고 공부 계획을 세웠다
0.000 | 교실에서 선생님께 질문했다
0.000 | 발표 준비를 친구와 함께 연습했다
0.000 | 오늘 운동장에서 축구를 했다
0.000 | 농구 골대 앞에서 친구들과 연습했다
0.000 | 점심으로 라면과 떡볶이를 먹었다
0.000 | 버스를 타고 학교에 갔다


## 5단계: Toy 임베딩 만들기 — 비슷한 뜻을 가까운 좌표로 놓기

이제 초보자용 핵심 비유로 돌아옵니다. 단어를 지도 위 좌표처럼 생각해봅시다.

여기서는 다섯 가지 축이 있다고 가정합니다.
- 시험/공부
- 발표/교실
- 스포츠
- 음식
- 이동수단

이 숫자는 **실제 모델이 만든 값이 아닙니다.** 학생이 임베딩의 원리를 눈으로 볼 수 있게 사람이 직접 설계한 toy embedding입니다. 실제 임베딩 모델은 이런 축을 사람이 정하지 않고 데이터에서 학습합니다.

축을 너무 적게 만들면 서로 다른 의미가 한 방향에 뭉쳐서 모든 것이 비슷해 보일 수 있습니다. 그래서 여기서는 `시험/공부`와 `발표/교실`을 분리해 차이가 더 잘 보이게 했습니다.

### 💬 ChatGPT에게 확인질문
```text
toy embedding과 실제 임베딩 모델의 차이를 설명해주세요.
사람이 직접 만든 5개 축짜리 예시 임베딩은 어떤 점에서 도움이 되고, 어떤 점에서 실제 AI 임베딩과 다른가요?
```


In [5]:
# 설명용 toy embedding: [시험/공부, 발표/교실, 스포츠, 음식, 이동수단]
toy_embeddings = {
    "시험": [0.95, 0.10, 0.00, 0.00, 0.00],
    "테스트": [0.92, 0.08, 0.00, 0.00, 0.00],
    "공부": [0.90, 0.15, 0.00, 0.00, 0.00],
    "계획": [0.75, 0.25, 0.00, 0.00, 0.05],
    "불안": [0.80, 0.10, 0.00, 0.00, 0.00],
    "발표": [0.20, 0.95, 0.00, 0.00, 0.00],
    "질문": [0.25, 0.80, 0.00, 0.00, 0.00],
    "학교": [0.35, 0.55, 0.00, 0.00, 0.45],
    "교실": [0.20, 0.90, 0.00, 0.00, 0.00],
    "선생님": [0.30, 0.85, 0.00, 0.00, 0.00],
    "친구": [0.15, 0.35, 0.25, 0.05, 0.00],
    "동아리": [0.10, 0.35, 0.45, 0.05, 0.00],
    "축구": [0.00, 0.00, 0.95, 0.00, 0.00],
    "농구": [0.00, 0.00, 0.92, 0.00, 0.00],
    "운동": [0.00, 0.00, 0.88, 0.00, 0.00],
    "운동장": [0.05, 0.05, 0.82, 0.00, 0.00],
    "골대": [0.00, 0.00, 0.85, 0.00, 0.00],
    "연습": [0.25, 0.45, 0.45, 0.00, 0.00],
    "라면": [0.00, 0.00, 0.00, 0.95, 0.00],
    "떡볶이": [0.00, 0.00, 0.00, 0.92, 0.00],
    "급식": [0.15, 0.05, 0.00, 0.78, 0.00],
    "도시락": [0.05, 0.05, 0.00, 0.82, 0.00],
    "버스": [0.00, 0.00, 0.00, 0.00, 0.95],
    "지하철": [0.00, 0.00, 0.00, 0.00, 0.92],
    "자동차": [0.00, 0.00, 0.00, 0.00, 0.90],
    "등교": [0.25, 0.20, 0.00, 0.00, 0.82],
    "지각": [0.20, 0.05, 0.00, 0.00, 0.85],
    "늦지": [0.10, 0.00, 0.00, 0.00, 0.88],
}

pairs = [
    ("시험", "테스트"),
    ("발표", "질문"),
    ("축구", "농구"),
    ("라면", "떡볶이"),
    ("버스", "지하철"),
    ("시험", "라면"),
]

print("📍 toy embedding 기반 단어 유사도")
for a, b in pairs:
    score = cosine_similarity(toy_embeddings[a], toy_embeddings[b])
    print(f"{a:>4} - {b:<4}: {score:.3f}")


📍 toy embedding 기반 단어 유사도
  시험 - 테스트 : 1.000
  발표 - 질문  : 0.995
  축구 - 농구  : 1.000
  라면 - 떡볶이 : 1.000
  버스 - 지하철 : 1.000
  시험 - 라면  : 0.000


## 6단계: 단어 개수 벡터와 Toy 임베딩 비교하기

이 단계가 가장 중요합니다.

단어 개수 벡터는 **같은 단어가 겹치는지**를 잘 봅니다. 반면 임베딩은 **다른 단어라도 의미가 가까운지**를 보려고 합니다.

아래 예시에서 `시험`과 `테스트`는 글자는 다르지만 의미가 가깝습니다. 이 차이를 두 방식으로 비교해봅니다.

### 💬 ChatGPT에게 확인질문
```text
단어 개수 벡터와 의미 임베딩의 차이를 "시험"과 "테스트" 예시로 설명해주세요.
왜 같은 단어가 없어도 의미 임베딩에서는 두 문장이 가깝게 나올 수 있나요?
```


In [6]:
def average_vectors(vectors, size=None):
    if size is None:
        size = len(next(iter(toy_embeddings.values())))
    if not vectors:
        return [0.0] * size
    return [sum(vector[i] for vector in vectors) / len(vectors) for i in range(size)]


def sentence_embedding(sentence):
    vectors = []
    matched_words = []
    for token in simple_tokenize(sentence):
        # 한국어 조사/어미가 붙은 경우를 위해 토큰 안에 포함된 기본 단어를 찾아봅니다.
        for word, vector in toy_embeddings.items():
            if word in token:
                vectors.append(vector)
                matched_words.append(word)
                break
    return average_vectors(vectors), matched_words


def embedding_only(sentence):
    return sentence_embedding(sentence)[0]

comparison_query = "테스트가 너무 힘들었다"
print("🔬 같은 질문을 두 방식으로 비교")
print("질문:", comparison_query)

print("\n[1] 단어 개수 벡터 방식")
query_count = count_vector(comparison_query, vocabulary)
for sentence in sentences[:4]:
    score = cosine_similarity(query_count, count_vector(sentence, vocabulary))
    print(f"{score:.3f} | {sentence}")

print("\n[2] toy embedding 방식")
query_emb, matched = sentence_embedding(comparison_query)
print("질문에서 찾은 임베딩 단어:", matched)
for sentence in sentences[:4]:
    score = cosine_similarity(query_emb, embedding_only(sentence))
    print(f"{score:.3f} | {sentence}")


🔬 같은 질문을 두 방식으로 비교
질문: 테스트가 너무 힘들었다

[1] 단어 개수 벡터 방식
0.354 | 오늘 시험이 너무 어려웠다
0.354 | 이번 테스트는 정말 힘들었다
0.000 | 시험 불안을 줄이려고 공부 계획을 세웠다
0.000 | 교실에서 선생님께 질문했다

[2] toy embedding 방식
질문에서 찾은 임베딩 단어: ['테스트']
1.000 | 오늘 시험이 너무 어려웠다
1.000 | 이번 테스트는 정말 힘들었다
0.996 | 시험 불안을 줄이려고 공부 계획을 세웠다
0.364 | 교실에서 선생님께 질문했다


## 7단계: 문장 임베딩 — 단어 벡터를 합쳐 문장 벡터 만들기

문장도 벡터로 만들 수 있습니다. 가장 쉬운 방법은 문장 안에 있는 단어 벡터를 평균내는 것입니다.

실제 모델은 문맥까지 반영해서 훨씬 정교하게 문장 임베딩을 만들지만, 아래 방식만으로도 검색과 추천의 기본 흐름을 볼 수 있습니다.

아래 출력에서는 어떤 단어가 임베딩에 사용됐는지도 함께 보여줍니다. 이 부분을 보면 toy embedding 검색이 왜 그런 결과를 냈는지 이해하기 쉽습니다.

### 💬 ChatGPT에게 확인질문
```text
문장 임베딩이 무엇인지 설명해주세요.
단어 임베딩 여러 개를 평균내서 문장 벡터를 만드는 간단한 방식과, 실제 모델이 문맥을 반영하는 방식의 차이도 알려주세요.
```


In [8]:
query = "시험이 정말 힘들었다"
query_emb, query_words = sentence_embedding(query)

print("🧭 toy embedding 기반 문장 유사도")
print("기준 문장:", query)
print("기준 문장에서 찾은 임베딩 단어:", query_words)
for sentence in sentences:
    sent_emb, sent_words = sentence_embedding(sentence)
    score = cosine_similarity(query_emb, sent_emb)
    print(f"{score:.3f} | {sentence} | 사용 단어: {sent_words}")


🧭 toy embedding 기반 문장 유사도
기준 문장: 시험이 정말 힘들었다
기준 문장에서 찾은 임베딩 단어: ['시험']
1.000 | 오늘 시험이 너무 어려웠다 | 사용 단어: ['시험']
1.000 | 이번 테스트는 정말 힘들었다 | 사용 단어: ['테스트']
0.997 | 시험 불안을 줄이려고 공부 계획을 세웠다 | 사용 단어: ['시험', '불안', '공부', '계획']
0.381 | 교실에서 선생님께 질문했다 | 사용 단어: ['교실', '선생님', '질문']
0.394 | 발표 준비를 친구와 함께 연습했다 | 사용 단어: ['발표', '친구', '연습']
0.000 | 오늘 운동장에서 축구를 했다 | 사용 단어: ['운동', '축구']
0.183 | 농구 골대 앞에서 친구들과 연습했다 | 사용 단어: ['농구', '골대', '친구', '연습']
0.000 | 점심으로 라면과 떡볶이를 먹었다 | 사용 단어: ['라면', '떡볶이']
0.263 | 버스를 타고 학교에 갔다 | 사용 단어: ['버스', '학교']


## 8단계: 미니 검색 엔진 — 질문과 가까운 문서 찾기

이제 코드 레벨로 한 단계 올라갑니다.

임베딩 검색 엔진의 기본 구조는 단순합니다.

1. 문서들을 벡터로 바꿔 저장한다
2. 질문도 벡터로 바꾼다
3. 질문 벡터와 문서 벡터의 유사도를 계산한다
4. 가장 점수가 높은 문서를 보여준다

이번에는 검색 결과뿐 아니라 각 문서가 어떤 의미 축에 가까운지도 같이 출력합니다.

### 💬 ChatGPT에게 확인질문
```text
임베딩을 이용한 검색 엔진이 어떻게 동작하는지 단계별로 설명해주세요.
질문과 문서를 각각 벡터로 바꾼 뒤 가장 가까운 문서를 찾는 과정을 예시와 함께 알려주세요.
```


In [9]:
labels = ["시험/공부", "발표/교실", "스포츠", "음식", "이동수단"]


def explain_vector(vector):
    return sorted(zip(labels, vector), key=lambda row: row[1], reverse=True)


def search_documents(query, docs, top_k=3):
    query_emb, query_words = sentence_embedding(query)
    results = []
    for title, text in docs.items():
        doc_emb, doc_words = sentence_embedding(text)
        score = cosine_similarity(query_emb, doc_emb)
        results.append({
            "score": score,
            "title": title,
            "text": text,
            "query_words": query_words,
            "doc_words": doc_words,
            "doc_vector": doc_emb,
        })
    return sorted(results, key=lambda row: row["score"], reverse=True)[:top_k]

queries = [
    "시험이 걱정될 때 어떻게 하지?",
    "축구 연습을 꾸준히 하고 싶어",
    "학교에 늦지 않게 가는 법",
]

for query in queries:
    print(f"\n🔎 질문: {query}")
    for result in search_documents(query, documents):
        print(f"- {result['score']:.3f} | {result['title']}: {result['text']}")
        print(f"  문서에서 찾은 단어: {result['doc_words']}")
        print(f"  문서 벡터 축: {explain_vector(result['doc_vector'])}")



🔎 질문: 시험이 걱정될 때 어떻게 하지?
- 0.997 | 공부 스트레스: 시험 기간에는 계획을 작게 나누고 쉬는 시간을 정하면 불안을 줄일 수 있다.
  문서에서 찾은 단어: ['시험', '계획', '불안']
  문서 벡터 축: [('시험/공부', 0.8333333333333334), ('발표/교실', 0.15), ('이동수단', 0.016666666666666666), ('스포츠', 0.0), ('음식', 0.0)]
- 0.394 | 발표 준비: 발표를 잘하려면 핵심 문장을 먼저 정하고 친구 앞에서 소리 내어 연습한다.
  문서에서 찾은 단어: ['발표', '친구', '연습']
  문서 벡터 축: [('발표/교실', 0.5833333333333334), ('스포츠', 0.2333333333333333), ('시험/공부', 0.19999999999999998), ('음식', 0.016666666666666666), ('이동수단', 0.0)]
- 0.130 | 점심 추천: 급식이 맞지 않는 날에는 간단하고 균형 잡힌 도시락을 준비할 수 있다.
  문서에서 찾은 단어: ['급식', '도시락']
  문서 벡터 축: [('음식', 0.8), ('시험/공부', 0.1), ('발표/교실', 0.05), ('스포츠', 0.0), ('이동수단', 0.0)]

🔎 질문: 축구 연습을 꾸준히 하고 싶어
- 0.939 | 운동 습관: 축구나 농구처럼 좋아하는 운동을 정해 꾸준히 하면 체력 관리에 도움이 된다.
  문서에서 찾은 단어: ['축구', '농구', '운동']
  문서 벡터 축: [('스포츠', 0.9166666666666666), ('시험/공부', 0.0), ('발표/교실', 0.0), ('음식', 0.0), ('이동수단', 0.0)]
- 0.650 | 발표 준비: 발표를 잘하려면 핵심 문장을 먼저 정하고 친구 앞에서 소리 내어 연습한다.
  문서에서 찾은 단어: ['발표', '친구', '연습']
  문서 벡터 축: [('발표/교실', 0.58333333333333

## 9단계: 실제 사례 — 꼬맨틀로 보는 단어 임베딩 게임

[꼬맨틀](https://semantle-ko.newsjel.ly/)은 오늘의 정답 단어를 맞히는 한국어 단어 유사도 게임입니다.

일반적인 단어 맞추기 게임은 철자나 글자 위치를 힌트로 주지만, 꼬맨틀은 **정답 단어와 내가 입력한 단어의 의미적 유사도**를 점수로 보여줍니다. 예를 들어 정답이 `학교`라면 `교실`, `선생님`, `수업`, `학생`처럼 같은 문맥에서 자주 등장할 법한 단어가 높은 점수를 받을 수 있습니다. 반대로 철자가 비슷해도 의미가 멀면 점수가 낮을 수 있습니다.

### 꼬맨틀이 임베딩을 쓰는 방식

꼬맨틀의 핵심 흐름은 이 노트북에서 만든 미니 검색 엔진과 거의 같습니다.

1. 각 단어를 임베딩 벡터로 바꿉니다.
2. 오늘의 정답 단어 벡터를 정합니다.
3. 사용자가 추측한 단어도 벡터로 바꿉니다.
4. 두 벡터의 유사도를 계산합니다.
5. 점수가 높을수록 정답과 의미적으로 가깝다고 알려줍니다.

꼬맨틀 공식 설명에 따르면, 유사도는 추측 단어와 정답 단어가 **의미맥락적으로 얼마나 비슷한지**를 -100에서 +100 사이 점수로 나타냅니다. 또한 유사도 추정을 위해 Common Crawl 및 Wikipedia 데이터로 사전 훈련된 **FastText**를 사용했다고 설명합니다.

### 왜 반대말도 가까울 수 있을까?

꼬맨틀에서 중요한 점은 “뜻이 정확히 같은가”보다 **비슷한 문맥에서 자주 쓰이는가**입니다.

예를 들어 `사랑`과 `증오`는 의미상 반대처럼 보일 수 있지만, 감정이나 인간관계 같은 비슷한 문맥에서 함께 등장할 수 있습니다. 그래서 임베딩 공간에서는 생각보다 가까울 수 있습니다.

또한 동음이의어도 중요합니다. 예를 들어 `밤`은 어두운 시간일 수도 있고 먹는 밤일 수도 있습니다. 어떤 문맥에서 학습되었는지에 따라 가까운 단어가 달라질 수 있습니다.

### 이 노트북과 연결하기

우리의 toy embedding에서는 사람이 직접 축을 만들었습니다.

```text
"시험"   → [0.95, 0.10, 0.00, 0.00, 0.00]
"테스트" → [0.92, 0.08, 0.00, 0.00, 0.00]
```

하지만 꼬맨틀은 사람이 직접 축을 정하지 않고, 대량의 한국어 텍스트에서 학습된 FastText 임베딩을 사용합니다.

즉, 이 노트북의 toy embedding과 `cosine_similarity`는 꼬맨틀의 핵심 아이디어를 아주 작게 흉내낸 것입니다. 실제 꼬맨틀은 훨씬 많은 단어와 훨씬 큰 차원의 벡터를 사용한다고 이해하면 됩니다.

### 직접 관찰해보기

1. [꼬맨틀](https://semantle-ko.newsjel.ly/)에 접속합니다.
2. 아무 단어 5개를 입력하고 유사도 점수를 기록합니다.
3. 점수가 높은 단어들이 어떤 공통 문맥을 갖는지 추측합니다.
4. 점수가 낮은 단어는 왜 멀게 나왔는지 설명해봅니다.
5. “철자가 비슷한 단어”와 “의미가 비슷한 단어” 중 무엇이 더 높은 점수를 받는지 비교합니다.

### 💬 ChatGPT에게 확인질문
```text
꼬맨틀이라는 단어 유사도 게임이 임베딩을 어떻게 활용하는지 설명해주세요.
정답 단어와 추측 단어를 벡터로 바꾸고, 코사인 유사도로 의미적 가까움을 계산한다는 흐름을 고등학생 눈높이로 알려주세요.
그리고 왜 반대말이나 같은 문맥의 단어가 가깝게 나올 수 있는지도 설명해주세요.
```

출처: [꼬맨틀 공식 페이지](https://semantle-ko.newsjel.ly/), [NewsJelly/semantle-ko GitHub](https://github.com/NewsJelly/semantle-ko)


## 10단계: 코드 레벨 — 재사용 가능한 임베딩 검색 클래스 만들기

이번에는 함수를 조금 더 정리해서 `MiniEmbeddingIndex` 클래스로 만듭니다.

이 코드는 작지만 실제 벡터 검색 시스템의 뼈대와 비슷합니다.
- `add`: 문서를 추가하고 벡터를 미리 계산
- `search`: 질문과 가까운 문서 반환
- `explain_vector`: 벡터가 어떤 축에 가까운지 설명

실제 서비스에서는 이 역할을 벡터 데이터베이스와 임베딩 모델이 더 큰 규모로 수행합니다.

### 💬 ChatGPT에게 확인질문
```text
MiniEmbeddingIndex 같은 작은 검색 클래스를 만든다고 할 때,
add, search, explain_vector 함수가 각각 어떤 역할을 하는지 고등학생도 이해할 수 있게 설명해주세요.
```


In [10]:
class MiniEmbeddingIndex:
    def __init__(self, embedding_fn, labels):
        self.embedding_fn = embedding_fn
        self.labels = labels
        self.items = []

    def add(self, title, text):
        vector, matched_words = self.embedding_fn(text)
        self.items.append({
            "title": title,
            "text": text,
            "vector": vector,
            "matched_words": matched_words,
        })

    def search(self, query, top_k=3):
        query_vector, query_words = self.embedding_fn(query)
        scored = []
        for item in self.items:
            score = cosine_similarity(query_vector, item["vector"])
            scored.append((score, query_words, item))
        return sorted(scored, key=lambda row: row[0], reverse=True)[:top_k]

    def explain_vector(self, vector):
        return sorted(zip(self.labels, vector), key=lambda row: row[1], reverse=True)

index = MiniEmbeddingIndex(sentence_embedding, labels)

for title, text in documents.items():
    index.add(title, text)

query = "발표와 시험 준비가 둘 다 걱정돼"
query_vector, query_words = sentence_embedding(query)
print("🧪 MiniEmbeddingIndex 검색")
print("질문:", query)
print("질문에서 찾은 단어:", query_words)
print("질문 벡터 설명:", index.explain_vector(query_vector))
print("\n검색 결과:")
for score, _, item in index.search(query, top_k=3):
    print(f"- {score:.3f} | {item['title']}: {item['text']}")
    print(f"  문서 단어: {item['matched_words']}")
    print(f"  문서 벡터 설명: {index.explain_vector(item['vector'])}")


🧪 MiniEmbeddingIndex 검색
질문: 발표와 시험 준비가 둘 다 걱정돼
질문에서 찾은 단어: ['발표', '시험']
질문 벡터 설명: [('시험/공부', 0.575), ('발표/교실', 0.525), ('스포츠', 0.0), ('음식', 0.0), ('이동수단', 0.0)]

검색 결과:
- 0.846 | 공부 스트레스: 시험 기간에는 계획을 작게 나누고 쉬는 시간을 정하면 불안을 줄일 수 있다.
  문서 단어: ['시험', '계획', '불안']
  문서 벡터 설명: [('시험/공부', 0.8333333333333334), ('발표/교실', 0.15), ('이동수단', 0.016666666666666666), ('스포츠', 0.0), ('음식', 0.0)]
- 0.820 | 발표 준비: 발표를 잘하려면 핵심 문장을 먼저 정하고 친구 앞에서 소리 내어 연습한다.
  문서 단어: ['발표', '친구', '연습']
  문서 벡터 설명: [('발표/교실', 0.5833333333333334), ('스포츠', 0.2333333333333333), ('시험/공부', 0.19999999999999998), ('음식', 0.016666666666666666), ('이동수단', 0.0)]
- 0.133 | 점심 추천: 급식이 맞지 않는 날에는 간단하고 균형 잡힌 도시락을 준비할 수 있다.
  문서 단어: ['급식', '도시락']
  문서 벡터 설명: [('음식', 0.8), ('시험/공부', 0.1), ('발표/교실', 0.05), ('스포츠', 0.0), ('이동수단', 0.0)]


## 11단계: RAG 흐름 — 찾은 문서를 답변 재료로 만들기

RAG는 Retrieval-Augmented Generation의 줄임말입니다.

쉽게 말하면 다음 순서입니다.

1. 질문과 관련 있는 문서를 먼저 찾는다
2. 찾은 문서를 AI에게 함께 준다
3. AI가 그 자료를 참고해 답한다

이 노트북의 목적은 임베딩 이해이므로, RAG는 “임베딩 검색이 어디에 쓰이는지”를 보여주는 응용 예시로만 다룹니다.

### 💬 ChatGPT에게 확인질문
```text
RAG가 무엇인지 아주 쉽게 설명해주세요.
임베딩 검색이 RAG에서 어떤 역할을 하는지, "질문과 관련 문서를 먼저 찾고 답변한다"는 흐름으로 알려주세요.
```


In [11]:
def make_rag_context(query, search_index, top_k=2):
    lines = []
    for score, query_words, item in search_index.search(query, top_k=top_k):
        lines.append(
            f"[{item['title']}] {item['text']} "
            f"(유사도 {score:.3f}, 질문 단어 {query_words}, 문서 단어 {item['matched_words']})"
        )
    return "\n".join(lines)

query = "시험 기간에 불안을 줄이고 발표도 준비하려면 어떻게 해야 해?"
context = make_rag_context(query, index)

print("🤖 RAG에 넣을 수 있는 참고 문맥")
print("질문:", query)
print("\n찾은 문서:")
print(context)
print("\n다음 단계에서는 이 문맥과 질문을 함께 LLM에 전달해 답변을 만들 수 있습니다.")


🤖 RAG에 넣을 수 있는 참고 문맥
질문: 시험 기간에 불안을 줄이고 발표도 준비하려면 어떻게 해야 해?

찾은 문서:
[공부 스트레스] 시험 기간에는 계획을 작게 나누고 쉬는 시간을 정하면 불안을 줄일 수 있다. (유사도 0.938, 질문 단어 ['시험', '불안', '발표'], 문서 단어 ['시험', '계획', '불안'])
[발표 준비] 발표를 잘하려면 핵심 문장을 먼저 정하고 친구 앞에서 소리 내어 연습한다. (유사도 0.710, 질문 단어 ['시험', '불안', '발표'], 문서 단어 ['발표', '친구', '연습'])

다음 단계에서는 이 문맥과 질문을 함께 LLM에 전달해 답변을 만들 수 있습니다.


## 12단계: 실습 과제 — 임베딩을 이해하기 위한 실험

아래 과제는 “검색 엔진 만들기”보다 **임베딩이 어떻게 의미를 표현하는지 이해하기**에 초점을 둡니다.

### 과제 1. 새 단어 추가하기
`toy_embeddings`에 새 단어 3개를 추가하세요.

예시:
- `숙제`: 공부/학교 축이 높게
- `배구`: 스포츠 축이 높게
- `김밥`: 음식 축이 높게

추가 후 `cosine_similarity`로 비슷한 단어와 정말 가까워졌는지 확인하세요.

### 과제 2. 일부러 잘못된 임베딩 넣어보기
`라면`의 벡터를 스포츠 쪽으로 바꿔보세요. 검색 결과가 어떻게 이상해지는지 관찰하세요.

이 과제의 핵심 질문:
```text
임베딩 숫자가 잘못되면 AI 검색 결과도 왜 이상해질까요?
```

### 과제 3. 단어 개수 방식과 임베딩 방식 비교하기
아래 두 문장을 비교하세요.

```text
오늘 시험이 어려웠다
이번 테스트는 힘들었다
```

단어 개수 벡터에서는 왜 덜 비슷하게 보이고, toy embedding에서는 왜 더 비슷하게 보이는지 설명해보세요.

### 과제 4. 나만의 작은 의미 축 설계하기
새로운 주제를 골라 축 4개를 직접 설계하세요.

| 주제 | 축 예시 |
|---|---|
| 음식 추천 | 매운맛, 건강함, 간편함, 든든함 |
| 진로 탐색 | 수학, 창의성, 소통, 탐구 |
| 운동 추천 | 팀플레이, 체력, 유연성, 경쟁 |
| 학교 생활 | 공부, 관계, 활동, 건강 |

마지막에는 아래 문장을 완성해보세요.

```text
내 toy embedding은 실제 임베딩과 다르다. 왜냐하면 ____________.
하지만 임베딩의 핵심 아이디어를 보여준다. 왜냐하면 ____________.
```

### 💬 ChatGPT에게 확인질문
```text
제가 toy embedding 실습을 하려고 합니다.
새 단어를 추가하고, 일부러 잘못된 벡터를 넣어보고, 검색 결과가 어떻게 달라지는지 관찰하는 실습 계획을 만들어주세요.
초보자가 따라 할 수 있게 단계별로 제시해주세요.
```


## 📝 정리: Beginner에서 Code Level까지

| 단계 | 배운 내용 | 코드에서 한 일 |
|---|---|---|
| 개념 | 임베딩은 의미의 숫자 좌표 | 5개 축 toy embedding으로 단어별 의미 좌표 만들기 |
| 토큰화 | 문장을 작은 조각으로 나눔 | `simple_tokenize` 구현 |
| 벡터화 | 글을 숫자 목록으로 바꿈 | 단어 개수 벡터 구현 |
| 유사도 | 가까운 의미를 점수로 계산 | `cosine_similarity` 구현 |
| 비교 | 단어 겹침과 의미 가까움의 차이 | count vector vs toy embedding 비교 |
| 문장 임베딩 | 단어 벡터를 합쳐 문장 표현 | 평균 벡터 구현 |
| 검색 | 질문과 가까운 문서 찾기 | 검색 결과와 벡터 축 설명 출력 |
| 실제 사례 | 꼬맨틀은 단어 임베딩 게임 | FastText와 의미 유사도 개념 연결 |
| 코드 구조화 | 재사용 가능한 검색 도구 | `MiniEmbeddingIndex` 구현 |
| RAG | 찾은 문서를 답변 재료로 사용 | 참고 문맥 생성 |

전체 흐름은 이렇게 기억하면 됩니다.

1. 텍스트 입력 → 2. 토큰화 → 3. 임베딩 → 4. 유사도 계산 → 5. 관련 문서 검색 → 6. 답변 생성

> 이 노트북의 숫자는 설명용 toy embedding입니다. 실제 임베딩은 사람이 직접 축을 정해 넣는 것이 아니라, 많은 데이터와 학습을 통해 만들어집니다. 그래도 핵심은 같습니다: **의미가 비슷하면 벡터도 가깝게 만든다.**